# SCI1070 — Semaine 11 : explorer, et mesurer un échec correctement

| Partie | Ce que vous y faites |
| --- | --- |
| Échauffement | River Swim : trois comptages, cinq graines |
| A | la méthode de référence, et **un échec correctement mesuré** |
| B | réseaux bruités contre calendrier de \(\varepsilon\) |
| C | une prime de nouveauté, et le piège qu'elle tend |
| Solution courte | deux références en six lignes, **après** vos volets |

**La tâche est `MountainCar-v0`, et ce n'est pas un choix de confort** : c'est la
tâche de ce cours sur laquelle la méthode \(\varepsilon\)-gloutonne échoue
**complètement**.

Deux exigences propres à cette semaine : **mesurez le temps jusqu'à la première
atteinte du but**, et **évaluez toujours sur un environnement sans prime**.

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import collections
import math

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sci1070.protocole import intervalle, se_recouvrent
from sci1070.courbes import tracer_bande
from sci1070.graines import fixer_graines

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRAINES = [0, 1, 2]

env = gym.make("MountainCar-v0")
print("MountainCar-v0 | obs", env.observation_space.shape, "| actions", env.action_space.n)
print("limite de pas :", gym.registry["MountainCar-v0"].max_episode_steps)
print("termine = le drapeau est atteint  |  tronque = les 200 pas sont écoulés")
env.close()

---
# Échauffement : reproduisez les trois nombres de River Swim

In [ ]:
class RiverSwim:
    """Une chaîne d'états. Le courant pousse vers la gauche ; le but est à droite."""

    def __init__(self, n_etats=6, graine=0):
        self.n_etats = n_etats
        self.rng = np.random.default_rng(graine)
        ...        # À COMPLÉTER : la Figure 11-1, écrite comme une table. Les rives n'ont qu'une action.

    def actions(self, etat):
        return list(self.transitions[etat])

    def tirer(self, etat, action):
        ...        # À COMPLÉTER : un tirage dans la ligne (etat, action) de la table

    def compter_visites(self, n_pas, pas_par_episode=10):
        """Agent PUREMENT aléatoire. Le tableau rendu somme exactement à n_pas."""
        ...        # À COMPLÉTER : le comptage, et la remise à l'état 1 tous les `pas_par_episode`


for n_pas in [100, 1_000, 10_000]:
    visites = RiverSwim(graine=1).compter_visites(n_pas)
    assert visites.sum() == n_pas, "la ligne ne somme pas au nombre de pas"
    print(f"{n_pas:6d} pas : {visites}  (somme {visites.sum()})")

In [ ]:
...        # À COMPLÉTER : question 1 : cinq graines, et l'amplitude du comptage du dernier état

In [ ]:
...        # À COMPLÉTER : questions 2 et 3, avec la prédiction écrite AVANT

---
# Volet A : la méthode de référence, et un échec correctement mesuré

> **Ce volet est le seul de tout le cours dont le résultat attendu est un échec, et
> je veux que vous le rapportiez comme une mesure.** Si après cent mille pas votre
> compteur d'atteintes du but affiche zéro, votre volet A est **réussi**.

In [ ]:
# La preuve que la distinction termine/tronque compte : un agent aléatoire.
env = gym.make("MountainCar-v0")
buts, tronques = 0, 0
for k in range(10):
    obs, _ = env.reset(seed=k)
    while True:
        obs, r, termine, tronque, _ = env.step(env.action_space.sample())
        if termine or tronque:
            buts += int(termine)
            tronques += int(tronque)
            break
env.close()
print(f"sur dix épisodes aléatoires : {buts} atteinte(s) du but, {tronques} troncature(s)")
print("-> si votre compteur affiche 10, vous comptez les troncatures")

In [ ]:
class Instruments:
    """Les deux instruments de la section 11.5. Installés avant l'entraînement."""

    def __init__(self):
        self.buts_cumules = 0
        self.positions_max = []
        self.premier_but_au_pas = None

    def episode(self, positions, termine, pas_entrainement):
        ...        # À COMPLÉTER : les deux instruments, et la date du premier but

In [ ]:
class CoucheBruitee(nn.Module):
    """Couche linéaire à bruit factorisé (section 6.9), reprise sans la réécrire."""

    def __init__(self, n_entree, n_sortie, sigma_initial=0.5):
        super().__init__()
        self.n_entree, self.n_sortie = n_entree, n_sortie
        borne = 1.0 / math.sqrt(n_entree)
        self.mu_poids = nn.Parameter(torch.empty(n_sortie, n_entree).uniform_(-borne, borne))
        self.sigma_poids = nn.Parameter(
            torch.full((n_sortie, n_entree), sigma_initial / math.sqrt(n_entree)))
        self.mu_biais = nn.Parameter(torch.empty(n_sortie).uniform_(-borne, borne))
        self.sigma_biais = nn.Parameter(
            torch.full((n_sortie,), sigma_initial / math.sqrt(n_entree)))
        self.register_buffer("eps_entree", torch.zeros(n_entree))
        self.register_buffer("eps_sortie", torch.zeros(n_sortie))
        self.retirer_bruit()

    @staticmethod
    def _echelle(x):
        return x.sign() * x.abs().sqrt()

    def retirer_bruit(self):
        self.eps_entree.copy_(self._echelle(torch.randn(self.n_entree)))
        self.eps_sortie.copy_(self._echelle(torch.randn(self.n_sortie)))

    def forward(self, x):
        if not self.training:
            return nn.functional.linear(x, self.mu_poids, self.mu_biais)
        poids = self.mu_poids + self.sigma_poids * self.eps_sortie.outer(self.eps_entree)
        biais = self.mu_biais + self.sigma_biais * self.eps_sortie
        return nn.functional.linear(x, poids, biais)


class ReseauQ(nn.Module):
    """Deux entrées, une couche cachée de 128, trois sorties. Minuscule."""

    def __init__(self, n_obs, n_actions, bruite=False):
        super().__init__()
        self.bruite = bruite
        self.corps = nn.Sequential(nn.Linear(n_obs, 128), nn.ReLU())
        if bruite:
            self.tete = CoucheBruitee(128, n_actions)
        else:
            self.tete = nn.Linear(128, n_actions)

    def forward(self, x):
        return self.tete(self.corps(x))

    def retirer_bruit(self):
        if self.bruite:
            self.tete.retirer_bruit()

In [ ]:
# La vérification de l'étape 2 du volet B : deux passes après un nouveau tirage.
reseau_bruite = ...        # À COMPLÉTER : votre réseau, tête bruitée
reseau_bruite.train()
obs_test = torch.zeros(1, 2)
sortie_1 = reseau_bruite(obs_test)
reseau_bruite.retirer_bruit()
sortie_2 = reseau_bruite(obs_test)
assert not torch.allclose(sortie_1, sortie_2), "le bruit n'est pas retiré"
print("les deux passes diffèrent : le bruit est bien retiré")

In [ ]:
def entrainer(variante="epsilon", graine=0, max_pas=100_000,
              frequence_bruit=1, c_prime=0.0, env_id="MountainCar-v0"):
    """UN programme, variantes en argument. `variante` ∈ {epsilon, bruite, rnd}."""
    ...        # À COMPLÉTER : votre réseau Q profond du jalon 2, calendrier de epsilon de 1,0 à 0,02


resultat_a = entrainer("epsilon", graine=0)
i = resultat_a["instruments"]
print(f"buts atteints : {i.buts_cumules}")
print(f"position maximale finale : {np.mean(i.positions_max[-50:]):.3f} "
      f"(le drapeau est à 0,5)")
print(f"epsilon a cessé d'explorer au pas {resultat_a['epsilon_fini_au_pas']}")

**Étape 4 — trois phrases.** Combien de fois le but a été atteint, ce que la position
maximale a fait, et **à quel pas votre \(\varepsilon\) a cessé d'explorer**. **La
troisième est celle qui compte** : après ce pas, l'agent ne joue plus que sa politique
gloutonne, laquelle n'a jamais vu de récompense non nulle. **L'échec devient
permanent et non seulement lent.**

---
# Volet B : des réseaux bruités contre un calendrier de ε

In [ ]:
resultat_b = entrainer("bruite", graine=0)
ib = resultat_b["instruments"]
print(f"{'variante':<12} | {'buts':>5} | {'premier but au pas':>19} | {'pos. max moyenne':>17}")
for nom, res in [("epsilon", resultat_a), ("bruité", resultat_b)]:
    inst = res["instruments"]
    print(f"{nom:<12} | {inst.buts_cumules:>5} | "
          f"{str(inst.premier_but_au_pas):>19} | "
          f"{np.mean(inst.positions_max[:50]):>17.3f}")

In [ ]:
...        # À COMPLÉTER : étape 4 : la fréquence de tirage du bruit, avec la prédiction écrite avant

---
# Volet C : une prime de nouveauté, et le piège qu'elle tend

**Deux environnements, pas un.** L'environnement d'**entraînement** porte la prime ;
celui d'**évaluation** ne la porte pas.

In [ ]:
for c in [0.1, 1.0, 10.0]:
    scores = ...        # À COMPLÉTER : trois graines
    moyenne, bas, haut = intervalle(scores)
    premiers = [entrainer("rnd", graine=g, c_prime=c)["instruments"].premier_but_au_pas
                for g in GRAINES]
    print(f"c = {c:5.1f} : retour {moyenne:8.1f}  [{bas:.1f}, {haut:.1f}]  "
          f"| premiers buts aux pas {premiers}")

In [ ]:
...        # À COMPLÉTER : étape 5 : la perte du prédicteur, et la marque du premier but

> **Quatre pièges, et chacun coûte une soirée.** `termine` contre `tronque` ; le
> coefficient \(c\) appliqué **deux fois** (votre agent devient un touriste) ;
> l'évaluation **sous enveloppe** (vos deux courbes montent ensemble et vous ne
> mesurez plus la tâche) ; et le **réseau de référence qui apprend** — sans
> `requires_grad_(False)`, les deux réseaux convergent l'un vers l'autre, la prime
> tombe à zéro partout, et votre méthode redevient celle du volet A, en plus lente.

---
# La solution courte

In [ ]:
# Une reference que vous n'avez pas ecrite, pour situer la votre.
# A executer APRES vos trois volets.
from stable_baselines3 import DQN, PPO
import gymnasium as gym

env = gym.make("MountainCar-v0")

# 1. La methode de valeur : attendez-vous a ce qu'elle n'arrive a rien.
dqn = DQN("MlpPolicy", env, verbose=0).learn(total_timesteps=100_000)

# 2. La methode de politique, SANS exploration dirigee : elle devrait au
#    moins trouver le but, comme la section 11.6 le rapporte.
ppo = PPO("MlpPolicy", env, verbose=0).learn(total_timesteps=200_000)

In [ ]:
...        # À COMPLÉTER : la mesure qui ferme la semaine : les DEUX grandeurs, pour A, B, C et SB3